<a href="https://colab.research.google.com/github/24x01a05r9-spec/task-management-application/blob/main/Task_Manager_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
from IPython.display import display, HTML

# Define the HTML, CSS (Tailwind), and Javascript codebase as a unified string
app_code = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Colab Task Workspace</title>
    <!-- Tailwind CSS for modern responsive styling -->
    <script src="jsdelivr.net"></script>
    <!-- FontAwesome for visual icon anchors -->
    <link rel="stylesheet" href="cloudflare.com">
</head>
<body class="bg-slate-900 text-slate-100 min-h-screen font-sans">

    <div class="max-w-6xl mx-auto p-4 md:p-8">
        <!-- Header Dashboard Section -->
        <header class="flex flex-col md:flex-row justify-between items-start md:items-center gap-4 mb-8 pb-6 border-b border-slate-800">
            <div>
                <h1 class="text-3xl font-black tracking-tight text-transparent bg-clip-text bg-gradient-to-r from-blue-400 to-indigo-400">
                    <i class="fa-solid fa-list-check mr-2"></i>TaskWorkspace
                </h1>
                <p class="text-slate-400 text-sm mt-1">Interactive task orchestrator running inside Google Colab</p>
            </div>

            <!-- Quick Progress Micro-Dashboard -->
            <div class="flex gap-4 w-full md:w-auto">
                <div class="bg-slate-800/60 border border-slate-700/50 rounded-xl p-3 flex-1 md:w-28 text-center">
                    <span class="block text-xs font-bold text-slate-400 uppercase tracking-wider">Total</span>
                    <span id="stat-total" class="text-2xl font-black text-blue-400">0</span>
                </div>
                <div class="bg-slate-800/60 border border-slate-700/50 rounded-xl p-3 flex-1 md:w-28 text-center">
                    <span class="block text-xs font-bold text-slate-400 uppercase tracking-wider">Pending</span>
                    <span id="stat-pending" class="text-2xl font-black text-amber-400">0</span>
                </div>
                <div class="bg-slate-800/60 border border-slate-700/50 rounded-xl p-3 flex-1 md:w-28 text-center">
                    <span class="block text-xs font-bold text-slate-400 uppercase tracking-wider">Done</span>
                    <span id="stat-done" class="text-2xl font-black text-emerald-400">0</span>
                </div>
            </div>
        </header>

        <!-- Main Workspace Layout Grid -->
        <div class="grid grid-cols-1 lg:grid-cols-3 gap-8">

            <!-- Left Column: Unified Configuration & Input Form -->
            <div class="lg:col-span-1 space-y-6">
                <div class="bg-slate-800/40 border border-slate-800 backdrop-blur-md rounded-2xl p-6 shadow-xl sticky top-6">
                    <h2 class="text-lg font-bold text-slate-200 mb-4 flex items-center gap-2">
                        <i class="fa-solid fa-circle-plus text-blue-400"></i>
                        <span id="form-title">Create New Task</span>
                    </h2>

                    <form id="task-form" class="space-y-4">
                        <input type="hidden" id="task-id">

                        <div>
                            <label class="block text-xs font-bold uppercase tracking-wider text-slate-400 mb-1">Task Title *</label>
                            <input type="text" id="task-title" required placeholder="e.g., Refactor API pipeline"
                                class="w-full bg-slate-900 border border-slate-700 rounded-xl px-4 py-2.5 text-sm text-slate-100 placeholder-slate-500 focus:outline-none focus:border-blue-500 transition-colors">
                        </div>

                        <div>
                            <label class="block text-xs font-bold uppercase tracking-wider text-slate-400 mb-1">Description</label>
                            <textarea id="task-desc" rows="3" placeholder="Add specific details or notes..."
                                class="w-full bg-slate-900 border border-slate-700 rounded-xl px-4 py-2.5 text-sm text-slate-100 placeholder-slate-500 focus:outline-none focus:border-blue-500 transition-colors resize-none"></textarea>
                        </div>

                        <div class="grid grid-cols-2 gap-4">
                            <div>
                                <label class="block text-xs font-bold uppercase tracking-wider text-slate-400 mb-1">Priority</label>
                                <select id="task-priority" class="w-full bg-slate-900 border border-slate-700 rounded-xl px-3 py-2.5 text-sm text-slate-100 focus:outline-none focus:border-blue-500 transition-colors">
                                    <option value="low">🟢 Low</option>
                                    <option value="medium" selected>🟡 Medium</option>
                                    <option value="high">🔴 High</option>
                                </select>
                            </div>
                            <div>
                                <label class="block text-xs font-bold uppercase tracking-wider text-slate-400 mb-1">Status</label>
                                <select id="task-status" class="w-full bg-slate-900 border border-slate-700 rounded-xl px-3 py-2.5 text-sm text-slate-100 focus:outline-none focus:border-blue-500 transition-colors">
                                    <option value="todo">To Do</option>
                                    <option value="in_progress">In Progress</option>
                                    <option value="done">Done</option>
                                </select>
                            </div>
                        </div>

                        <div class="pt-2 flex gap-3">
                            <button type="submit" id="submit-btn" class="flex-1 bg-gradient-to-r from-blue-500 to-indigo-600 hover:from-blue-600 hover:to-indigo-700 text-white text-sm font-bold py-3 px-4 rounded-xl transition-all shadow-lg shadow-blue-500/10 flex items-center justify-center gap-2 cursor-pointer">
                                <i class="fa-solid fa-floppy-disk"></i> Save Task
                            </button>
                            <button type="button" id="cancel-btn" class="hidden bg-slate-700 hover:bg-slate-600 text-slate-200 text-sm font-bold py-3 px-4 rounded-xl transition-all cursor-pointer">
                                Cancel
                            </button>
                        </div>
                    </form>
                </div>
            </div>

            <!-- Right Column: Interactive Search Filter & Responsive Task Boards -->
            <div class="lg:col-span-2 space-y-6">
                <!-- Search & Filters Segment -->
                <div class="bg-slate-800/40 border border-slate-800 rounded-2xl p-4 flex flex-col sm:flex-row gap-4 items-center justify-between shadow-md">
                    <div class="relative w-full sm:w-72">
                        <i class="fa-solid fa-magnifying-glass absolute left-3.5 top-1/2 -translate-y-1/2 text-slate-500 text-sm"></i>
                        <input type="text" id="search-bar" placeholder="Filter tasks instantly..."
                            class="w-full bg-slate-900 border border-slate-700 rounded-xl pl-10 pr-4 py-2 text-sm text-slate-100 placeholder-slate-500 focus:outline-none focus:border-blue-500 transition-colors">
                    </div>

                    <div class="flex gap-2 w-full sm:w-auto overflow-x-auto pb-1 sm:pb-0">
                        <button onclick="filterTasks('all')" class="filter-tab px-4 py-1.5 rounded-lg text-xs font-bold uppercase tracking-wider bg-blue-500 text-white transition-all cursor-pointer" data-filter="all">All</button>
                        <button onclick="filterTasks('todo')" class="filter-tab px-4 py-1.5 rounded-lg text-xs font-bold uppercase tracking-wider bg-slate-800 text-slate-400 hover:bg-slate-700 transition-all cursor-pointer" data-filter="todo">To Do</button>
                        <button onclick="filterTasks('in_progress')" class="filter-tab px-4 py-1.5 rounded-lg text-xs font-bold uppercase tracking-wider bg-slate-800 text-slate-400 hover:bg-slate-700 transition-all cursor-pointer" data-filter="in_progress">In Progress</button>
                        <button onclick="filterTasks('done')" class="filter-tab px-4 py-1.5 rounded-lg text-xs font-bold uppercase tracking-wider bg-slate-800 text-slate-400 hover:bg-slate-700 transition-all cursor-pointer" data-filter="done">Done</button>
                    </div>
                </div>

                <!-- Dynamic Reactive Task Container -->
                <div id="tasks-container" class="space-y-3 min-h-[300px]">
                    <!-- Injection point for dynamic items -->
                </div>
            </div>

        </div>
    </div>

    <!-- Script Application Logic Core -->
    <script>
        // State Initialization backed by local browser caching
        let tasks = JSON.parse(localStorage.getItem('colab_tasks')) || [
            { id: '1', title: 'Verify Colab runtime container connectivity', desc: 'Confirm that frontend HTML/JS nodes are communicating flawlessly inside the notebook context.', priority: 'high', status: 'done' },
            { id: '2', title: 'Design modular full-stack architecture configurations', desc: 'Incorporate database tier schemas along with secure structural routing boundaries.', priority: 'medium', status: 'in_progress' },
            { id: '3', title: 'Implement real-time updates feature branch', desc: 'Integrate WebSockets hooks to handle cross-client state changes asynchronously.', priority: 'low', status: 'todo' }
        ];

        let activeFilter = 'all';
        let searchQuery = '';

        const taskForm = document.getElementById('task-form');
        const tasksContainer = document.getElementById('tasks-container');
        const searchBar = document.getElementById('search-bar');

        // CRUD: Render Loop
        function renderTasks() {
            tasksContainer.innerHTML = '';

            const filtered = tasks.filter(task => {
                const matchesFilter = activeFilter === 'all' || task.status === activeFilter;
                const matchesSearch = task.title.toLowerCase().includes(searchQuery.toLowerCase()) ||
                                      task.desc.toLowerCase().includes(searchQuery.toLowerCase());
                return matchesFilter && matchesSearch;
            });

            updateMetrics();

            if (filtered.length === 0) {
                tasksContainer.innerHTML = `
                    <div class="text-center py-16 bg-slate-800/20 border border-dashed border-slate-800 rounded-2xl">
                        <i class="fa-solid fa-folder-open text-slate-600 text-4xl mb-3"></i>
                        <p class="text-slate-400 text-sm font-medium">No matching tasks found in this scope.</p>
                    </div>
                `;
                return;
            }

            filtered.forEach(task => {
                const card = document.createElement('div');
                card.className = `p-5 bg-slate-800/50 border border-slate-800/80 rounded-2xl flex items-start justify-between gap-4 hover:border-slate-700 transition-all shadow-sm group ${task.status === 'done' ? 'opacity-60' : ''}`;

                // Color badges mapping config
                const priorityColors = { low: 'bg-emerald-500/10 text-emerald-400 border-emerald-500/20', medium: 'bg-amber-500/10 text-amber-400 border-amber-500/20', high: 'bg-rose-500/10 text-rose-400 border-rose-500/20' };
                const statusLabels = { todo: 'To Do', in_progress: 'In Progress', done: 'Completed' };
                const statusStyles = { todo: 'bg-slate-700 text-slate-300', in_progress: 'bg-blue-600 text-white', done: 'bg-emerald-600 text-white' };

                card.innerHTML = `
                    <div class="space-y-2 flex-1 min-w-0">
                        <div class="flex flex-wrap items-center gap-2">
                            <span class="text-xs px-2.5 py-0.5 rounded-full font-bold uppercase tracking-wider border ${priorityColors[task.priority]}">
                                ${task.priority}
                            </span>
                            <span class="text-[10px] px-2 py-0.5 rounded-md font-extrabold uppercase tracking-wide ${statusStyles[task.status]}">
                                ${statusLabels[task.status]}
                            </span>
                        </div>
                        <h3 class="text-base font-bold text-slate-100 break-words ${task.status === 'done' ? 'line-through text-slate-500' : ''}">
                            ${escapeHtml(task.title)}
                        </h3>
                        ${task.desc ? `<p class="text-sm text-slate-400 line-clamp-2 break-words">${escapeHtml(task.desc)}</p>` : ''}
                    </div>
                    <div class="flex items-center gap-1 shrink-0 opacity-100 sm:opacity-0 group-hover:opacity-100 transition-opacity">
                        <button onclick="toggleStatus('${task.id}')" class="p-2 hover:bg-slate-700 rounded-xl text-slate-400 hover:text-emerald-400 transition-colors cursor-pointer" title="Toggle Status Completion">
                            <i class="fa-solid ${task.status === 'done' ? 'fa-circle-check text-emerald-400' : 'fa-circle'}"></i>
                        </button>
                        <button onclick="startEdit('${task.id}')" class="p-2 hover:bg-slate-700 rounded-xl text-slate-400 hover:text-blue-400 transition-colors cursor-pointer" title="Edit Task parameters">
                            <i class="fa-solid fa-pen-to-square"></i>
                        </button>
                        <button onclick="deleteTask('${task.id}')" class="p-2 hover:bg-slate-700 rounded-xl text-slate-400 hover:text-rose-400 transition-colors cursor-pointer" title="Delete Task permanently">
                            <i class="fa-solid fa-trash"></i>
                        </button>
                    </div>
                `;
                tasksContainer.appendChild(card);
            });
        }

        // CRUD: Form Submit Route Processing logic
        taskForm.addEventListener('submit', (e) => {
            e.preventDefault();
            const id = document.getElementById('task-id').value;
            const title = document.getElementById('task-title').value.trim();
            const desc = document.getElementById('task-desc').value.trim();
            const priority = document.getElementById('task-priority').value;
            const status = document.getElementById('task-status').value;

            if(!title) return;

            if (id) {
                // Modify existing entry
                tasks = tasks.map(t => t.id === id ? { ...t, title, desc, priority, status } : t);
            } else {
                // Construct brand new item tracking record
                const newTask = { id: Date.now().toString(), title, desc, priority, status };
                tasks.unshift(newTask);
            }

            saveAndSync();
            resetForm();
        });

        // CRUD: Auxiliary interactions implementation closures
        window.toggleStatus = (id) => {
            tasks = tasks.map(t => {
                if (t.id === id) {
                    const nextStatus = t.status === 'done' ? 'todo' : 'done';
                    return { ...t, status: nextStatus };
                }
                return t;
            });
            saveAndSync();
        };

        window.startEdit = (id) => {
            const task = tasks.find(t => t.id === id);
            if (!task) return;

            document.getElementById('task-id').value = task.id;
            document.getElementById('task-title').value = task.title;
            document.getElementById('task-desc').value = task.desc;
            document.getElementById('task-priority').value = task.priority;
            document.getElementById('task-status').value = task.status;

            document.getElementById('form-title').innerText = "Modify Task Parameters";
            document.getElementById('submit-btn').innerHTML = `<i class="fa-solid fa-file-pen"></i> Update Task`;
            document.getElementById('cancel-btn').classList.remove('hidden');
            document.getElementById('task-title').focus();
        };

        window.deleteTask = (id) => {
            tasks = tasks.filter(t => t.id !== id);
            saveAndSync();
            if(document.getElementById('task-id').value === id) resetForm();
        };

        document.getElementById('cancel-btn').addEventListener('click', resetForm);

        function resetForm() {
            taskForm.reset();
            document.getElementById('task-id').value = '';
            document.getElementById('form-title').innerText = "Create New Task";
            document.getElementById('submit-btn').innerHTML = `<i class="fa-solid fa-floppy-disk"></i> Save Task`;
            document.getElementById('cancel-btn').classList.add('hidden');
        }

        window.filterTasks = (filterValue) => {
            activeFilter = filterValue;
            document.querySelectorAll('.filter-tab').forEach(btn => {
                if(btn.getAttribute('data-filter') === filterValue) {
                    btn.className = "filter-tab px-4 py-1.5 rounded-lg text-xs font-bold uppercase tracking-wider bg-blue-500 text-white transition-all cursor-pointer";
                } else {
                    btn.className = "filter-tab px-4 py-1.5 rounded-lg text-xs font-bold uppercase tracking-wider bg-slate-800 text-slate-400 hover:bg-slate-700 transition-all cursor-pointer";
                }
            });
            renderTasks();
        };

        searchBar.addEventListener('input', (e) => {
            searchQuery = e.target.value;
            renderTasks();
        });

        function updateMetrics() {
            document.getElementById('stat-total').innerText = tasks.length;
            document.getElementById('stat-pending').innerText = tasks.filter(t => t.status !== 'done').length;
            document.getElementById('stat-done').innerText = tasks.filter(t => t.status === 'done').length;
        }

        function saveAndSync() {
            localStorage.setItem('colab_tasks', JSON.stringify(tasks));
            renderTasks();
        }

        function escapeHtml(text) {
            const div = document.createElement('div');
            div.innerText = text;
            return div.innerHTML;
        }

        // Bootstrapping execution context
        renderTasks();
    </script>
</body>
</html>
"""

# Render the application cleanly within the output pane interface layout
display(HTML(app_code))